# Transaction Foundation Model on Ray — Part 3: Tokenize — the static/dynamic split

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

In Part 2 we built the canonical dataset and the temporal split for evaluation. Now, tokenize - we turn each card's raw transactions into the padded, per-field token windows the model trains on — as a stateless `groupby("card_id").map_groups(...)` that runs across the cluster, one card at a time.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                                        # same knob as Part 2; mini = tiny, CPU-only
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

with open(paths["splits"]) as f:
    splits = json.load(f)                            # temporal cutoffs from Part 2

2026-06-25 17:45:54,406	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.113.42:6379...
2026-06-25 17:45:54,432	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at https://session-nridlis6nyy2hi9qv4il5lcj6q.i.anyscaleuserdata.com 
/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## Encoding static & dynamic fields

As we discussed in Part 2, we handle static fields (like card type) in a different way than dynamic fields (like amount). 

In comparison, NVIDIA's early TFM blueprint flattens every transaction into ~12 tokens in one shared vocabulary, so a history of *N* transactions costs ~12*N* positions. We split the fields by how they behave over a card's sequence:

- **Static** fields (`issuer`, `card_type`, `bin_region`, `home_state`) describe the *card* and never change within its history. They're embedded **once** and added to every position — they never spend sequence length.
- **Dynamic** fields (`amount`, `merchant`, `mcc`, `hour`, `day_of_week`) describe each *transaction*. Each gets its own embedding table, and a transaction becomes **one** position whose vector is the sum of its field embeddings.

So *N* transactions is *N* positions, not ~12*N* — cheaper, and a stronger inductive bias because the model isn't re-learning that a card's issuer is constant. At `full` scale 512 positions cover 512 transactions, versus ~315 in the blueprint's 4096-token context.

## Turn each card's history into token sequences

We now turn each card's raw transactions into the fixed-length integer sequences the transformer will train on. This writes two datasets: pretraining windows and labeled eval samples.

Our approach is to group transactions by card_id and run a tokenization function per card with map_groups, since a card's tokens depend on its whole ordered history. The tokenizer cuts that transaction history into fixed-length windows and computes the gap between consecutive transactions. Using map_groups along with the stateless tokenizer means this code scales trivially with Ray.

Two configuration parameters:

- `normal_keep` comes from `eval_normal_keep`: it keeps every fraud and downsamples normals toward `target_eval_samples`. Each kept normal is stamped with a weight of `1 / normal_keep` so the downsampling cancels out in the metrics (Part 6).
- `num_partitions` sizes the group-by shuffle. Ray defaults to 200, which would scatter the output into hundreds of tiny files at this scale.

In one pass, this tokenization function returns two kinds of row per card in the one pass, distinguished by a `kind` column:
- **pretrain** — non-overlapping windows over the card's train-period history, for masked-feature pretraining (Part 4); never includes val/test transactions.
- **eval** — one window per target transaction (ending at it), labeled with that transaction's `is_fraud` and tagged `train`/`val`/`test` by its timestamp.

After `materialize`, the cell filters on `kind` and writes the two sets to separate Parquet directories (`PRETRAIN_DROP` strips the eval-only columns from the pretrain set), then `write_vocab` records the fixed vocabulary spec the model reads. 

In [2]:
from ray.data.expressions import col
from src.tokenizer import (
    make_tokenize_group_fn, eval_normal_keep, write_vocab, PRETRAIN_DROP,
)

tk = cfg["tokenize"]
seq_len = tk["seq_len"]
# Keep all frauds + enough normals to hit the eval-size target; the weight
# attached per sample undoes this downsampling in the metrics later.
normal_keep = eval_normal_keep(splits, tk["target_eval_samples"])
print(f"tokenizing at seq_len={seq_len}, normal_keep={normal_keep:.4f}")

# Re-runs reuse the cached token shards.
if not (os.path.exists(paths["tokenized_eval"]) and os.path.exists(paths["vocab"])):
    ds = ray.data.read_parquet(paths["raw"])

    # The per-card tokenizer UDF: windowing + bucketing/hashing live in
    # src/tokenizer.py (dataset-specific, deterministic). scripts/02_tokenize.py
    # composes this exact function, so the walkthrough and the job can't drift.
    group_fn = make_tokenize_group_fn(
        seq_len,
        train_end=splits["train_end"],            # pretrain windows: train-period only
        val_end=splits["val_end"],                # eval split tag by target timestamp
        normal_keep=normal_keep,
        holdout_keep=tk["holdout_keep"],
        max_pretrain_windows=tk["max_pretrain_windows"],
    )

    # One card = one group = one independent unit of work. No global shuffle.
    tokenized = (
        ds.groupby("card_id", num_partitions=tk["shuffle_partitions"])  # right-size the shuffle (Ray defaults to 200)
          .map_groups(group_fn, batch_format="numpy")
          .materialize()
    )

    # Two output streams from the one pass, split on the `kind` column.
    tokenized.filter(expr=col("kind") == "pretrain").drop_columns(PRETRAIN_DROP) \
        .write_parquet(paths["tokenized_pretrain"])
    tokenized.filter(expr=col("kind") == "eval").drop_columns(["kind"]) \
        .write_parquet(paths["tokenized_eval"])
    write_vocab(paths["vocab"])                       # the deterministic vocab spec the model reads

    print(f"  wrote {tokenized.count():,} token windows (pretrain + eval)")
    print(f"    pretrain -> {paths['tokenized_pretrain']}")
    print(f"    eval     -> {paths['tokenized_eval']}")
    print(f"    vocab    -> {paths['vocab']}")
else:
    print(f"  reusing cached tokens at {os.path.dirname(paths['tokenized_eval'].rstrip('/'))}/")

tokenizing at seq_len=32, normal_keep=0.0050
  reusing cached tokens at /mnt/cluster_storage/transaction-fm/tokenized/mini/


## The tokenized output

At `mini` scale this yields about **12k pretrain windows** and **~49k eval samples** — which include **every one of the 9,685 frauds** in the dataset, plus downsampled normals, tagged `train`/`val`/`test` by the temporal split. Each normal eval sample carries a weight of ~200 (= 1 / `normal_keep`) so the downsampling cancels out in the metrics; frauds carry weight 1.

One eval sample makes the split concrete. We decode the tokens back to human values for readability — under the hood every field is an integer id into that field's embedding table, and `merchant`/`mcc` are hashed, so they stay as bucket numbers. The **static** fields are a single value each, constant across the whole window; the **dynamic** fields are one row per transaction, and the window ends at the labeled (`target`) transaction.

In [ ]:
import glob
import pyarrow.parquet as pq
from src.tokenizer import decode_static_fields, decode_dynamic_window

def _parquet_rows(path):
    """Row count straight from the Parquet footers — no data scan."""
    return sum(pq.ParquetFile(f).metadata.num_rows
               for f in glob.glob(path.rstrip("/") + "/*.parquet"))

# This cell only reports counts and decodes ONE sample, so it never needs the
# per-field token arrays in memory. Reading the whole tokenized dir back through
# pandas (pd.read_parquet(dir)) would pull every seq_len-wide token vector onto
# the head node, single-threaded — minutes at `small`, an OOM at `full`. Instead:
# row counts come from the Parquet metadata, and the split/fraud breakdown reads
# only the two scalar columns it touches (projection pushdown skips the arrays).
n_pre = _parquet_rows(paths["tokenized_pretrain"])
meta = pd.read_parquet(paths["tokenized_eval"], columns=["split", "label"])
by = meta["split"].value_counts()

print(f"{n_pre:,} pretrain windows")
print(f"{len(meta):,} eval samples  ({int((meta['label'] == 1).sum()):,} fraud · "
      f"train {by.get('train', 0):,} / val {by.get('val', 0):,} / test {by.get('test', 0):,})")
print(f"seq_len {seq_len} — one position per transaction\n")

# Decode one sample back to human values so the static/dynamic split is legible.
# Read a single shard and take its first row — not the whole eval set.
one_shard = sorted(glob.glob(paths["tokenized_eval"].rstrip("/") + "/*.parquet"))[0]
row = pd.read_parquet(one_shard).iloc[0]
print(f"One eval sample — card {int(row['card_id'])}, label {int(row['label'])}, "
      f"{int(row['length'])} real transactions\n")
print("STATIC (one value per field, added to every position):")
print("  " + "   ".join(f"{k}={v}" for k, v in decode_static_fields(row).items()))
print("\nDYNAMIC (one transaction per row, decoded from tokens; 'target' is the labeled txn):")
print(decode_dynamic_window(row).to_string())

The vocabulary is fixed, so the model knows every embedding-table size up front — no data scan, no cold-start. Each field's table has `PAD`/`MASK`/`OOV` reserved ids plus its own value space (16 amount buckets, 2000 hashed merchant buckets, and so on).

In [4]:
from src.tokenizer import field_vocab_sizes, STATIC_FIELDS, DYNAMIC_FIELDS

sizes = field_vocab_sizes()
print("embedding rows per field (includes PAD/MASK/OOV):")
for f in STATIC_FIELDS:
    print(f"  static   {f:<18} {sizes[f]:>6,}")
for f in DYNAMIC_FIELDS:
    print(f"  dynamic  {f:<18} {sizes[f]:>6,}")

embedding rows per field (includes PAD/MASK/OOV):
  static   issuer                 16
  static   card_type              10
  static   bin_region             10
  static   home_state             57
  dynamic  amount_bucket          19
  dynamic  merchant_bucket     2,003
  dynamic  merchant_category      12
  dynamic  mcc                   131
  dynamic  hour                   27
  dynamic  day_of_week            10


## Takeaways

**Ray Data**
- Tokenization is a `groupby("card_id").map_groups(fn)`: one card is one group, and `fn` needs the whole history to window it and compute inter-transaction gaps.
- Because the vocabulary is deterministic, `fn` is **stateless** — no global aggregation, no cross-card coordination, so every group tokenizes independently across the cluster. The same code runs on one CPU at `mini` and across the cluster at `full`.
- `num_partitions` right-sizes the hash shuffle; Ray's default (200) is tuned for far larger clusters and would scatter the output into hundreds of tiny files at demo scale.
- The notebook composes the same `make_tokenize_group_fn` that `scripts/02_tokenize.py` runs headless, so the walkthrough and the batch job produce identical tokens.

**The tokenizer**
- The static/dynamic split makes a history of *N* transactions *N* positions, not ~12*N* — statics are embedded once instead of repeated at every step.
- One pass emits two streams: train-period **pretrain** windows and per-transaction **eval** samples (all fraud kept, normals downsampled with importance weights).
- Positions are time-aware (the log-bucketed gap since the previous transaction), and amounts are log-bucketed rather than fed as a raw scalar.

---

## Next

**Part 4 — Pretrain with Ray Train**: masked-feature modeling over the pretrain windows, plain PyTorch wrapped by Ray Train for DDP, sharding, and checkpointing — same code from 1 CPU worker to N GPUs.